# 🚀 LIGHTGBM CLASSIFIER (GRADIENT BOOSTING)

> **🎯 AMAÇ:** XGBoost'tan daha hızlı, büyük veri için optimize edilmiş gradient boosting  
> **📥 GİRDİ:** X_train, X_test, y_train, y_test (bellekten)  
> **📤 ÇIKTI:** y_pred, Accuracy, Confusion Matrix, Classification Report, Feature Importance  

### ⏱️ NE ZAMAN KULLANILIR?
- Büyük veri setlerinde (XGBoost'tan 10x+ daha hızlı)
- Yüksek feature sayısında (GPU desteği ile)
- Kaggle'da önde gelen modellerden biri
- Kategorik feature'ları native olarak işleyebilir

### 🔧 TEMEL PARAMETRELER
| Parametre | Varsayılan | Açıklama |
|-----------|-----------|----------|
| `n_estimators` | 100 | Ağaç sayısı |
| `learning_rate` | 0.1 | Adım büyüklüğü |
| `max_depth` | -1 | Derinlik (-1 = sınırsız) |
| `num_leaves` | 31 | **En kritik parametre** — ağaç karmaşıklığı |
| `min_child_samples` | 20 | Yaprak min. örnek (overfit kontrolü) |
| `subsample` | 1.0 | Satır örnekleme oranı |
| `colsample_bytree` | 1.0 | Feature örnekleme oranı |
| `reg_alpha` | 0.0 | L1 regularization |
| `reg_lambda` | 0.0 | L2 regularization |
| `class_weight` | None | Dengesiz veri için `'balanced'` |
| `boosting_type` | 'gbdt' | `'gbdt'`, `'dart'`, `'goss'` |

### 🆚 LightGBM vs XGBoost
| Özellik | LightGBM | XGBoost |
|---------|----------|---------| 
| Hız | ⚡⚡⚡ | ⚡⚡ |
| Büyük veri | ✅ | 🔶 |
| Kategorik destek | ✅ Native | ❌ |

---
### ⚠️ UYARI
X_train, X_test, y_train, y_test bellekte olmali.  
Kurulum: `pip install lightgbm`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import warnings; warnings.filterwarnings('ignore')
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
print('=' * 55)
print('🚀 LIGHTGBM CLASSIFIER BAŞLATILIYOR')
print('=' * 55)
print(f'LightGBM version: {lgb.__version__}')
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# 1. MODEL TANIMI
lgb_clf = lgb.LGBMClassifier(
    n_estimators      = 300,       # agac sayisi
    learning_rate     = 0.05,      # kucuk lr + fazla agac = guclu model
    max_depth         = -1,        # -1: sinursuz (num_leaves ile kontrol)
    num_leaves        = 31,        # KRITIK: 2^max_depth'ten kucuk tut
    min_child_samples = 20,        # overfit kontrolu
    subsample         = 0.8,       # satir ornekleme
    colsample_bytree  = 0.8,       # feature ornekleme
    reg_alpha         = 0.1,       # L1
    reg_lambda        = 0.1,       # L2
    class_weight      = 'balanced',# dengesiz veri (yoksa None yap)
    boosting_type     = 'gbdt',    # 'gbdt', 'dart', 'goss'
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1
)

y_train_fit = y_train.values.ravel() if hasattr(y_train, 'values') else y_train
y_test_fit  = y_test.values.ravel()  if hasattr(y_test,  'values') else y_test
lgb_clf.fit(X_train, y_train_fit, eval_set=[(X_test, y_test_fit)],
            callbacks=[lgb.log_evaluation(period=50)])
print('\n✅ LightGBM modeli egitildi.')

In [ ]:
# 3. TAHMİN & METRİKLER
y_pred = lgb_clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'\n🎯 Accuracy: %{acc*100:.2f}')
print('\n📋 Classification Report:')
print('-' * 55)
print(classification_report(y_test, y_pred))

In [ ]:
# 4. CONFUSION MATRIX
cm      = confusion_matrix(y_test, y_pred)
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', linewidths=0.5,
            annot_kws={'size': 14, 'weight': 'bold'}, ax=axes[0])
axes[0].set_title(f'Confusion Matrix (Ham)\nAccuracy: %{acc*100:.2f}', fontweight='bold')
axes[0].set_xlabel('Tahmin'); axes[0].set_ylabel('Gercek')
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='YlOrRd',
            linewidths=0.5, vmin=0, vmax=1, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalize)', fontweight='bold')
axes[1].set_xlabel('Tahmin'); axes[1].set_ylabel('Gercek')
plt.suptitle('🚀 LightGBM - Confusion Matrix Analizi', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# 5. FEATURE IMPORTANCE (SPLIT + GAIN)
feature_names = list(X_train.columns) if hasattr(X_train,'columns') else [f'f{i}' for i in range(X_train.shape[1])]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (imp_type, color) in zip(axes, [('split','#2196F3'),('gain','#4CAF50')]):
    importances = lgb_clf.feature_importances_ if imp_type == 'split' else lgb_clf.booster_.feature_importance('gain')
    df_imp = pd.DataFrame({'feature': feature_names, 'score': importances}).sort_values('score', ascending=False).head(15)
    cmap = plt.cm.Blues if imp_type == 'split' else plt.cm.Greens
    shades = cmap(np.linspace(0.4, 0.9, len(df_imp)))[::-1]
    ax.barh(df_imp['feature'][::-1], df_imp['score'][::-1], color=shades, edgecolor='white')
    ax.set_title(f"{'Kullanum Sayisi (Split)' if imp_type=='split' else 'Ort. Kazanc (Gain)'}", fontweight='bold')
plt.suptitle('🚀 LightGBM - Feature Importance', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# 6. CROSS-VALIDATION
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)
cv_scores = cross_val_score(cv_model, X_train, y_train_fit, cv=skf, scoring='accuracy', n_jobs=-1)
print('📊 5-Fold CV:')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {"█" * int(s*30)} %{s*100:.2f}')
print(f'\n🎯 CV Ort: %{cv_scores.mean()*100:.2f} | Std: ±%{cv_scores.std()*100:.2f}')
print(f'📌 Test Acc: %{acc*100:.2f}')

In [ ]:
# 7. GRIDSEARCHCV
param_grid = {
    'n_estimators'      : [100, 300],
    'num_leaves'        : [15, 31, 63],   # LIGHTGBM'in kritik parametresi
    'learning_rate'     : [0.05, 0.1],
    'min_child_samples' : [10, 20, 50],
    'subsample'         : [0.7, 1.0],
}
grid = GridSearchCV(
    lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
    param_grid,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='accuracy', n_jobs=-1, verbose=1
)
print('🔍 GridSearchCV baslatiliyor...')
grid.fit(X_train, y_train_fit)
print('\n✅ En iyi parametreler:')
for k, v in grid.best_params_.items(): print(f'  {k:<22}: {v}')
best_lgb  = grid.best_estimator_
best_acc  = accuracy_score(y_test, best_lgb.predict(X_test))
print(f'\n🎯 CV best: %{grid.best_score_*100:.2f} | Test best: %{best_acc*100:.2f}')
print(f'📈 Iyilesme: %{(best_acc - acc)*100:.2f}')
print('\n✅ Kullanima hazir: best_lgb')